# 11-phase1-head-plasticity

**neuron 패러다임 Phase 1** — Attention module 단위 dynamic add 검증. 기존 GraphLM Phase 1 (block-level) 의 dead block 패턴이 head-level (정확히는 attention module level) 에서도 재현되는지가 첫 가설.

- **검증할 가설**:
  1. **Function preservation 학습 곡선상 검증** — α=0 으로 attention module 추가 직후 loss spike 없음
  2. **Dead attention 패턴 측정** — 추가된 attention 의 α 가 0 근처에 갇히는가? (기존 Phase 1 의 dead block 처럼)
  3. **Block 단위 vs Head 단위 dead 비율 비교** — fine-grained 가 dead 회피에 도움 되는가?
- **데이터**: TinyShakespeare (기존 Phase 1 과 동일)
- **시드**: 42
- **작성일**: 2026-05-25
- **연관**: Issue [#41](https://github.com/EinSofINTEREST/GraphLM/issues/41) / 비교 baseline PR [#38](https://github.com/EinSofINTEREST/GraphLM/pull/38) (기존 Phase 1 의 dead block 결과)

## 1. 환경 / 시드 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import NeuronConfig, NeuronGrowingDecoder, add_attn_function_preserving
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")
    torch.cuda.reset_peak_memory_stats(device=DEVICE)

## 2. 실험 설정

기존 GraphLM Phase 1 (`10-phase1-growing-decoder.ipynb`) 과 **동일한 hidden_dim / FFN / max_steps / lr** 사용. 차이는:
- `n_layers=4` 고정 (기존은 4→8 growth)
- `n_init_attn=1` 으로 시작, 학습 중 attention 추가 (round-robin 으로 block 0 → 1 → 2 → 3 → 0 → ...)
- `max_total_attn=8` — 기존 Phase 1 의 max_layers=8 과 같은 total 노드 수 (비교 균등화)

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
BLOCK_SIZE = 128

BACKBONE = dict(
    hidden_dim=256,
    n_heads=4,
    ffn_dim=1024,
    n_layers=4,
    n_init_attn=1,
)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,  # 4 blocks × 1 init + 4 growths = 8 total
    trigger_window=100,
    trigger_epsilon=0.04,  # 기존 #3 권장값
    trigger_cooldown=150,
    trigger_min_history=100,
)

for k, v in {**BACKBONE, **TRAIN}.items():
    print(f"  {k:22s} = {v}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
data_iter = iter_random_batches(dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=SEED)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 모델 초기화

In [ ]:
cfg = NeuronConfig(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=BLOCK_SIZE,
    **BACKBONE,
)
model = NeuronGrowingDecoder(cfg).to(DEVICE)

print(f"n_attn_per_block (init) : {model.n_attn_per_block}")
print(f"n_params (init)         : {model.n_params:,}")

## 5. 학습 (inline loop with round-robin attn add)

PlateauTrigger fire 시 다음 block 에 attention module 추가 (round-robin). 새 attention 의 param 은 `optimizer.add_param_group` 으로 등록 (기존 momentum 보존).

In [ ]:
trigger = PlateauTrigger(
    window=TRAIN["trigger_window"],
    epsilon=TRAIN["trigger_epsilon"],
    cooldown=TRAIN["trigger_cooldown"],
    min_history=TRAIN["trigger_min_history"],
)
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

losses = []
grow_events = []  # (step, target_block, total_attn_after)
next_block_to_grow = 0

model.train()
V = cfg.vocab_size
for step in range(1, TRAIN["max_steps"] + 1):
    x, y = next(data_iter)
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer.zero_grad()
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if trigger.update(loss.item()) and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]:
        target_block = next_block_to_grow
        add_attn_function_preserving(model, target_block)
        # 새 param 만 optimizer 에 등록 (기존 AdamW momentum 보존)
        block = model.blocks[target_block]
        new_params = (
            list(block.attn_lns[-1].parameters())
            + list(block.attns[-1].parameters())
            + [block.attn_alphas[-1]]
        )
        optimizer.add_param_group({"params": new_params})
        grow_events.append((step, target_block, sum(model.n_attn_per_block)))
        next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

print("=" * 50)
print(f"final n_attn_per_block : {model.n_attn_per_block} (total {sum(model.n_attn_per_block)})")
print(f"final n_params         : {model.n_params:,}")
print(f"grow events            : {len(grow_events)}")
for s, b, t in grow_events:
    print(f"  step={s:>5}  block={b}  total_attn={t}")
n = min(100, len(losses))
first = sum(losses[:n]) / n if n > 0 else 0.0
last = sum(losses[-n:]) / n if n > 0 else 0.0
print(f"loss first {n} avg : {first:.4f}")
print(f"loss last  {n} avg : {last:.4f}")
print(f"improvement       : {first - last:+.4f}")
if str(DEVICE).startswith("cuda"):
    print(f"GPU peak alloc    : {torch.cuda.max_memory_allocated(device=DEVICE) / 1e9:.3f} GB")

## 6. 손실 곡선 + grow event 마커

빨간 수직선이 attention 추가 시점. α=0 init 이라 spike 없어야 정상.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(1, len(losses) + 1), losses, lw=0.7, color="steelblue", label="train loss")
for s, b, _t in grow_events:
    ax.axvline(s, color="red", alpha=0.5, lw=0.9)
    ax.text(s, max(losses), f" b{b}", color="red", fontsize=7, va="top")
ax.set_xlabel("step")
ax.set_ylabel("CE loss")
ax.set_title("neuron Phase 1 — TinyShakespeare with head-level dynamic add")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curve.png", dpi=120)
plt.show()

## 7. block 별 attention α 분포 — dead attention 검출

기존 GraphLM Phase 1 의 결과: 초기 block α ≈ 0.96, 추가 block α ≈ 0.0016 (mean ratio 0.002 = init 의 0.2%).
본 neuron Phase 1 의 head-level 에서 같은 패턴인지, 다른 양상인지 정량 비교.

In [ ]:
print(f"{'block':>5}  {'attn_idx':>9}  {'is_init':>8}  {'alpha':>+10}  {'|alpha|':>9}")
print("-" * 55)
all_init = []  # 초기 attn (n_init_attn 안의 것) 의 |α|
all_added = []  # 추가 attn 의 |α|
for bi, block in enumerate(model.blocks):
    for ai, alpha_param in enumerate(block.attn_alphas):
        a = alpha_param.item()
        is_init = ai < cfg.n_init_attn
        (all_init if is_init else all_added).append(abs(a))
        print(f"{bi:>5}  {ai:>9}  {'init' if is_init else 'added':>8}  {a:>+10.4f}  {abs(a):>9.4f}")

## 8. dead attention 비율 + 기존 Phase 1 비교

In [ ]:
import statistics

DEAD_THRESHOLD = 0.05  # |α| < 0.05 이면 dead 로 판정

# 통계 inline 계산 — 노트북 규약 (로직 정의 금지) 준수
for label, values in [
    ("초기 attention (α=1.0 에서 시작)", all_init),
    ("추가 attention (α=0.0 에서 시작)", all_added),
]:
    print(f"{label} — |α| 통계:")
    if not values:
        print("  (none)")
    else:
        sigma = statistics.stdev(values) if len(values) > 1 else 0
        print(
            f"  n={len(values):>2}  mean={statistics.mean(values):.4f}  "
            f"min={min(values):.4f}  max={max(values):.4f}  σ={sigma:.4f}"
        )
    print()

if all_init and all_added:
    init_mean = statistics.mean(all_init)
    if init_mean > 0:
        ratio = statistics.mean(all_added) / init_mean
        print(f"비율 (mean |α_added| / mean |α_init|) : {ratio:.3f} ({ratio:.1%})")

n_dead = sum(1 for a in all_added if a < DEAD_THRESHOLD)
n_total_added = len(all_added)
if n_total_added > 0:
    print()
    print(
        f"dead attention (|α| < {DEAD_THRESHOLD}) : {n_dead}/{n_total_added} ({n_dead / n_total_added:.1%})"
    )

print()
print("=" * 55)
print("기존 GraphLM Phase 1 (block-level) 비교 — PR #38 결과:")
print("  초기 block mean |α|  : ~0.96")
print("  추가 block mean |α|  : ~0.0016")
print("  비율                 : 0.002 (0.2%)")
print("  dead block 비율      : 4/4 = 100% (모두 dead)")

## 9. 시각화 — block 위치별 attention |α| bar plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x_positions = []
heights = []
colors = []
labels = []
x = 0
for bi, block in enumerate(model.blocks):
    for ai, alpha_param in enumerate(block.attn_alphas):
        x_positions.append(x)
        heights.append(abs(alpha_param.item()))
        is_init = ai < cfg.n_init_attn
        colors.append("steelblue" if is_init else "orange")
        labels.append(f"b{bi}.a{ai}")
        x += 1
    x += 0.5  # block 사이 gap
ax.bar(x_positions, heights, color=colors, alpha=0.85)
ax.axhline(
    DEAD_THRESHOLD,
    color="red",
    linestyle="--",
    lw=1,
    alpha=0.5,
    label=f"dead threshold ({DEAD_THRESHOLD})",
)
ax.set_xticks(x_positions)
ax.set_xticklabels(labels, rotation=45, fontsize=8)
ax.set_ylabel("|alpha| (final)")
ax.set_title("neuron Phase 1 — attention |alpha| per (block, attn). 파란=init, 주황=added")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(OUT_DIR / "alpha_per_attn.png", dpi=120)
plt.show()

## 결과 요약 / 다음 가설

**검증된 것**:
- (가설 1) function preservation 학습 곡선상 검증 → §6 의 빨간 수직선 위치에서 loss spike 부재 시 통과
- (가설 2) Dead attention 패턴 측정 → §8 의 dead 비율 (|α| < 0.05) 출력
- (가설 3) Block vs Head 단위 비교 → §8 마지막 표 (기존 Phase 1 100% dead vs head-level X%)

**예상 결과 시나리오**:
- **A. head-level 도 dead 동일 (100% dead)** → 함수 단위 granularity 만으로는 dead 회피 불가. 다음 phase 에서 plasticity / Hebbian init 등 더 정교한 메커니즘 필요.
- **B. head-level 에서 dead 감소** → fine-grained granularity 가 도움. function-level 컨셉의 첫 긍정 증거.
- **C. head-level 에서 일부 active** → 가장 흥미로운 경우. 어떤 block 의 head 가 살아나는지 패턴 분석 → Csordás 2021 의 발견 (attention 가장 modular) 의 재현 또는 반박.

**다음 작업 후보 (neuron Phase 2)**:
- Differentiable Plasticity 의 α connection-level 적용 → α 가 weight matrix entry 별로 학습
- SMGrNN 의 local SPM trigger → loss 만이 아닌 local activation/gradient 기반 trigger
- Csordás 2021 의 differentiable weight mask 학습 → 어떤 attention 이 dead 인지 mask 로 학습